# 06b Rarity Analysis

## Purpose

This notebook is me checking whether the weaker macro scores are mostly a rare-token problem.

What I want to do here:
- load the saved class-level outputs from notebook 6
- group token classes by support in the masked test set
- see how F1, top-1 accuracy, AP, and AUC change as support gets smaller
- save a simple rarity summary that I can reuse later in slides or writeups


## Setup note

Same setup split again.

- code and notebooks stay in GitHub
- rarity outputs stay in Drive
- Colab pulls the repo first so the notebook uses the current `src/rarity_analysis.py` helper


In [ ]:
# ==============================================================================
# 0. SET UP THE COLAB ENVIRONMENT
# ==============================================================================
import os
import sys

from google.colab import drive

drive.mount('/content/drive')

GITHUB_OWNER = 'hb791-dev'
REPO_NAME = 'glycan-roberta'
REPO_URL = f'https://github.com/{GITHUB_OWNER}/{REPO_NAME}.git'
REPO_DIR = f'/content/{REPO_NAME}'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f'Reusing existing repo at {REPO_DIR}')
    !git -C {REPO_DIR} pull --ff-only

%cd {REPO_DIR}
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)


In [ ]:
# ==============================================================================
# 1. IMPORT HELPERS
# ==============================================================================
import importlib
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

import src.rarity_analysis as rarity_analysis

importlib.reload(rarity_analysis)

from src.rarity_analysis import (
    add_rarity_flags,
    assign_support_bins,
    build_rare_token_table,
    build_rarity_bin_summary,
    compute_rarity_summary,
    load_rarity_inputs,
    merge_rarity_tables,
    plot_metric_by_support_bin,
    plot_support_distribution,
    plot_support_metric_scatter,
    save_rarity_outputs,
)


## Pick the run

This is the main place I should edit when I want to switch models. I want the tokenizer family, setting label, and experiment name to stay lined up so I do not accidentally read the wrong results folder.


In [ ]:
# ==============================================================================
# 2. CONFIG
# ==============================================================================
PROJECT_ROOT = Path('/content/drive/MyDrive/ProjectRoot')

TOKENIZER_FAMILY = 'glyberta'
SETTING_LABEL = 'v1_train_only'
EXPERIMENT_NAME = 'mlm15_L6_H512_A8_lr00001_ep100_setv1_train_only'

SUPPORT_BINS = [
    (1, 9),
    (10, 24),
    (25, 99),
    (100, None),
]

RARE_SUPPORT_MAX = 24

print(f'Tokenizer family: {TOKENIZER_FAMILY}')
print(f'Setting label: {SETTING_LABEL}')
print(f'Experiment name: {EXPERIMENT_NAME}')


In [ ]:
# ==============================================================================
# 3. PATHS
# ==============================================================================
TEST_EVAL_DIR = (
    PROJECT_ROOT
    / 'results'
    / 'test_evaluation'
    / TOKENIZER_FAMILY
    / EXPERIMENT_NAME
)

RARITY_RESULTS_DIR = (
    PROJECT_ROOT
    / 'results'
    / 'rarity'
    / TOKENIZER_FAMILY
    / EXPERIMENT_NAME
)
RARITY_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TEST_SUMMARY_PATH = TEST_EVAL_DIR / 'test_summary.json'
PER_CLASS_METRICS_PATH = TEST_EVAL_DIR / 'per_class_metrics.csv'
TOP1_ROC_PER_CLASS_PATH = TEST_EVAL_DIR / 'top1_correctness_roc_per_class.csv'
TOP1_PR_PER_CLASS_PATH = TEST_EVAL_DIR / 'top1_correctness_pr_per_class.csv'

SUPPORT_DISTRIBUTION_PATH = RARITY_RESULTS_DIR / 'support_distribution.png'
SUPPORT_VS_F1_PATH = RARITY_RESULTS_DIR / 'support_vs_f1_scatter.png'
SUPPORT_VS_AP_PATH = RARITY_RESULTS_DIR / 'support_vs_ap_scatter.png'
SUPPORT_VS_AUC_PATH = RARITY_RESULTS_DIR / 'support_vs_auc_scatter.png'
METRIC_BY_SUPPORT_BIN_PATH = RARITY_RESULTS_DIR / 'metric_by_support_bin.png'

print(f'Test evaluation dir: {TEST_EVAL_DIR}')
print(f'Rarity results dir: {RARITY_RESULTS_DIR}')


## What support means here

I want to keep the rarity definition simple and tied to the outputs I already have.

Here, support means: how many times a token showed up as the true masked answer in the held-out test set.

So if a token only appears a handful of times in `y_true`, that class is going to have a much noisier estimate and it is also probably one of the harder places for macro metrics to drop.


In [ ]:
# ==============================================================================
# 4. LOAD THE SAVED NOTEBOOK-6 OUTPUTS
# ==============================================================================
loaded = load_rarity_inputs(
    test_summary_path=TEST_SUMMARY_PATH,
    per_class_metrics_path=PER_CLASS_METRICS_PATH,
    top1_roc_per_class_path=TOP1_ROC_PER_CLASS_PATH,
    top1_pr_per_class_path=TOP1_PR_PER_CLASS_PATH,
)

test_summary = loaded['test_summary']
per_class_metrics = loaded['per_class_metrics']
top1_roc_per_class = loaded['top1_roc_per_class']
top1_pr_per_class = loaded['top1_pr_per_class']

# Quick sanity check so I know the saved notebook-6 outputs loaded.
input_overview = pd.DataFrame([
    {'table': 'per_class_metrics', 'rows': len(per_class_metrics), 'columns': len(per_class_metrics.columns)},
    {'table': 'top1_roc_per_class', 'rows': len(top1_roc_per_class), 'columns': len(top1_roc_per_class.columns)},
    {'table': 'top1_pr_per_class', 'rows': len(top1_pr_per_class), 'columns': len(top1_pr_per_class.columns)},
])
display(input_overview)


## Merge the class-level tables

Notebook 6 already did the hard part. Here I just want one combined table so I can look at support, F1, top-1 accuracy, AP, and AUC together without bouncing between files.

I am also not showing all three raw input tables anymore because that was taking up space without helping me interpret much.


In [ ]:
# ==============================================================================
# 5. BUILD ONE MERGED RARITY TABLE
# ==============================================================================
merged_metrics = merge_rarity_tables(
    per_class_metrics=per_class_metrics,
    top1_roc_per_class=top1_roc_per_class,
    top1_pr_per_class=top1_pr_per_class,
)
merged_metrics = assign_support_bins(merged_metrics, SUPPORT_BINS)
merged_metrics = add_rarity_flags(merged_metrics, rare_support_max=RARE_SUPPORT_MAX)

print(f'Merged table has {len(merged_metrics)} token classes.')
print(f"{int(merged_metrics['is_rare'].sum())} classes fall in the rare bucket (support <= {RARE_SUPPORT_MAX}).")

expected_columns = [
    'token_id',
    'token',
    'support',
    'precision',
    'recall',
    'f1',
    'correct_count',
    'incorrect_count',
    'top1_accuracy',
    'auc',
    'average_precision',
]
missing_display_columns = [column_name for column_name in expected_columns if column_name not in merged_metrics.columns]

if missing_display_columns:
    print('Missing expected columns:', missing_display_columns)
else:
    print('Merged table has the columns I expected for the rarity pass.')


In [ ]:
# ==============================================================================
# 6. SIMPLE TABLES I ACTUALLY WANT TO LOOK AT
# ==============================================================================
rarity_bin_summary = build_rarity_bin_summary(merged_metrics)
rare_token_table = build_rare_token_table(merged_metrics, rare_support_max=RARE_SUPPORT_MAX)

if 'share_of_token_classes' not in rarity_bin_summary.columns:
    total_classes = rarity_bin_summary['num_token_classes'].sum()
    if total_classes > 0:
        rarity_bin_summary['share_of_token_classes'] = rarity_bin_summary['num_token_classes'] / total_classes
    else:
        rarity_bin_summary['share_of_token_classes'] = 0.0

summary_display = rarity_bin_summary[[
    'support_bin',
    'num_token_classes',
    'share_of_token_classes',
    'median_support',
    'mean_f1',
    'mean_top1_accuracy',
    'mean_average_precision',
    'mean_auc',
]].copy()
summary_display['share_of_token_classes'] = summary_display['share_of_token_classes'].round(3)
summary_display['mean_f1'] = summary_display['mean_f1'].round(3)
summary_display['mean_top1_accuracy'] = summary_display['mean_top1_accuracy'].round(3)
summary_display['mean_average_precision'] = summary_display['mean_average_precision'].round(3)
summary_display['mean_auc'] = summary_display['mean_auc'].round(3)

print('Support-bin summary')
display(summary_display)

print(f'Rare-token spotlight (support <= {RARE_SUPPORT_MAX})')
display(rare_token_table.head(20))


## Plots

The main thing I want to see is whether performance tends to rise with support.

I changed the support plot to use the same bins I am already using in the tables because the raw histogram was not helping much.

So now the blue chart is just context for how many token classes landed in each bin, and the green chart is the actual performance pattern across those same bins. The scatter plots also stay on raw support and mark the rare-token cutoff directly.


In [ ]:
# ==============================================================================
# 7. PLOTS
# ==============================================================================
plot_support_distribution(merged_metrics, SUPPORT_DISTRIBUTION_PATH)
plot_support_metric_scatter(merged_metrics, 'f1', SUPPORT_VS_F1_PATH, rare_support_max=RARE_SUPPORT_MAX)
plot_support_metric_scatter(merged_metrics, 'average_precision', SUPPORT_VS_AP_PATH, rare_support_max=RARE_SUPPORT_MAX)
plot_support_metric_scatter(merged_metrics, 'auc', SUPPORT_VS_AUC_PATH, rare_support_max=RARE_SUPPORT_MAX)
plot_metric_by_support_bin(rarity_bin_summary, 'mean_f1', METRIC_BY_SUPPORT_BIN_PATH)


In [ ]:
# ==============================================================================
# 8. BUILD A SMALL SUMMARY I CAN REUSE LATER
# ==============================================================================
rarity_summary = compute_rarity_summary(
    merged_metrics,
    test_summary=test_summary,
    rare_support_max=RARE_SUPPORT_MAX,
)

display(pd.DataFrame.from_dict(rarity_summary, orient='index', columns=['value']))


In [ ]:
# ==============================================================================
# 9. SAVE OUTPUTS
# ==============================================================================
rarity_config = {
    'tokenizer_family': TOKENIZER_FAMILY,
    'setting_label': SETTING_LABEL,
    'experiment_name': EXPERIMENT_NAME,
    'support_bins': SUPPORT_BINS,
    'rare_support_max': RARE_SUPPORT_MAX,
    'source_paths': {
        'test_summary_path': str(TEST_SUMMARY_PATH),
        'per_class_metrics_path': str(PER_CLASS_METRICS_PATH),
        'top1_roc_per_class_path': str(TOP1_ROC_PER_CLASS_PATH),
        'top1_pr_per_class_path': str(TOP1_PR_PER_CLASS_PATH),
    },
}

saved_paths = save_rarity_outputs(
    output_dir=RARITY_RESULTS_DIR,
    merged_df=merged_metrics,
    bin_summary_df=rarity_bin_summary,
    rare_token_df=rare_token_table,
    rarity_summary=rarity_summary,
    rarity_config=rarity_config,
)

for label, path in saved_paths.items():
    print(f'{label}: {path}')

print(f'support_distribution_path: {SUPPORT_DISTRIBUTION_PATH}')
print(f'support_vs_f1_path: {SUPPORT_VS_F1_PATH}')
print(f'support_vs_ap_path: {SUPPORT_VS_AP_PATH}')
print(f'support_vs_auc_path: {SUPPORT_VS_AUC_PATH}')
print(f'metric_by_support_bin_path: {METRIC_BY_SUPPORT_BIN_PATH}')


## Notes to myself

If this ends up showing a big support-performance trend, then the next thing I probably want is a cross-tokenizer comparison version so I can line up all four tokenizers in one view instead of reopening this notebook four separate times.
